# Optional Regression Module: Multiple Regression with Our Class Survey

**CP201A Planning Methods Gateway, Fall 2026 · Work at your own pace**

This module is **optional**. It's for students who want to try the advanced option in Part III of the final memo: a multivariate regression. You can do very well on the memo without it. If you'd like to stretch, plan on about two hours.

**What you'll learn**
- How a regression line summarizes the relationship between two variables (Section 3)
- How regression handles a dummy X and a category X, and what a "reference group" is (Sections 4 and 5)
- What "holding other variables constant" means, and why it matters for survey data (Section 6)
- How to use a 0/1 outcome, and report it in percentage points (Section 7)
- What to check before you trust a regression (Section 8)
- How to run your own, make a table and a chart, and write it up (Sections 9 and 10)

**Before you start:** finish Lab 12 and save `survey_cleaned_2026.csv`. Have your two P/NP #11 hypotheses handy: a good regression for your memo usually starts from one of them and asks whether the relationship holds once you account for a few other things.

**Reading (optional, on the syllabus):** Bueno de Mesquita, Ethan, and Anthony Fowler (2021). "Regression for Describing and Forecasting," in *Thinking Clearly with Data*, pp. 74 to 93.

Exercises are numbered `# EXERCISE #1`, `# EXERCISE #2`, and so on. **What you submit:** nothing from this notebook. If you use a regression in your memo, it goes in Part III as one of your tested findings, with a table or chart and a Data Note.

## 1. Setup: load the data

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.formula.api as smf

pd.set_option('display.float_format', '{:.2f}'.format)

Run the next cell. It looks for **your** cleaned file from Lab 12 (`survey_cleaned_2026.csv`), first in this folder and then in your Lab 12 folder. If it finds it, you're set. If not, it cleans the raw class file the same way Lab 12 does, using the same variable names, so the examples still run.

> **Use your own cleaned file for your own tests.** If the message below says it cleaned the raw file, you're looking at the *demo's* cleaning choices, not yours. That's fine for following along. For P/NP #11 and your final memo, finish Lab 12, save `survey_cleaned_2026.csv`, and rerun this cell until it says **"Loaded YOUR cleaned Lab 12 file."**

The last line makes `city4`, with Richmond and El Cerrito combined, the same way we did on November 18.

In [ ]:
def clean_raw_survey(raw):
    """The Lab 12 cleaning steps, condensed. See Lab 12 for what each step does and why."""
    d = raw[raw['Unnamed: 0'] != 'Example'].drop(columns=['Unnamed: 0'])
    d = d[d['City'].notna()].reset_index(drop=True)
    d = d.map(lambda v: v.strip() if isinstance(v, str) else v)
    d['survey_id'] = d['Surveyor'] + '_' + d['Survey Number'].astype(str)
    d = d.rename(columns={
        'Interviewer or self-administered': 'mode', 'City': 'city', 'Survey Language': 'language',
        'What best describes your role here?': 'role', 'What type of place is this?': 'place_type',
        'About how many years has this business or organization been at this location?': 'years_at_location',
        'About how many years have you worked or volunteered here?': 'years_worked',
        'Including you, about how many people work here?': 'num_workers',
        'Does this business or organization own or rent its space?': 'space_tenure',
        'Do you live in this neighborhood?': 'lives_here', 'How many years have you lived here?': 'years_lived',
        'Do you own or rent your home?': 'home_tenure',
        'On a typical workday, about how many minutes does it take you to get here?': 'commute_min',
        'How do you usually get here?': 'travel_mode',
        'This neighborhood has changed a lot in the past five years.': 'changed_a_lot',
        'Changes in this neighborhood in the past five years have been good for long-time residents.': 'good_for_longtime',
        'Changes in this neighborhood in the past five years have been good for businesses and organizations like this one.': 'good_for_business',
        'Long-time residents can afford to stay in this neighborhood.': 'afford_to_stay',
        'I support building more housing in this neighborhood.': 'support_housing',
        'I feel connected to the people and community in this neighborhood.': 'connected',
        'I feel safe walking alone at night in this neighborhood.': 'safe_at_night',
        'The number of customers or clients who come here': 'p12_customers',
        "This place's sales, revenue, or funding": 'p12_sales',
        "Rent for this business or organization's space": 'p12_space_rent',
        'Other costs of running this place (supplies, wages, utilities)': 'p12_other_costs',
        'Your own rent or mortgage payment': 'p12_own_housing',
        'Prices of everyday goods and services in this neighborhood': 'p12_prices'})

    def looks_like_date(x):
        return isinstance(x, str) and (re.match(r'^\d{4}-\d{2}-\d{2}', x) is not None
                                       or re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}', x) is not None)
    def to_years(x):
        if pd.isna(x): return np.nan
        if isinstance(x, (int, float)): return float(x)
        s = str(x).lower()
        if looks_like_date(s) or 'know' in s or s in ['n/a', 'na', 'skipped', 'did not answer']: return np.nan
        m = re.match(r'^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*$', s)
        if m: return (float(m.group(1)) + float(m.group(2))) / 2
        total, found = 0, False
        for number, unit in re.findall(r'(\d+(?:\.\d+)?)\s*(year|yr|month|mo|week|day)?', s):
            found = True
            v = float(number)
            total += v / 12 if unit.startswith('mo') else v / 52 if unit == 'week' else v / 365 if unit == 'day' else v
        return total if found else np.nan
    def to_minutes(x):
        if pd.isna(x): return np.nan
        if isinstance(x, (int, float)): return float(x)
        s = str(x).lower()
        if looks_like_date(s) or 'know' in s or 'skip' in s: return np.nan
        m = re.match(r'^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)', s)
        if m: v = (float(m.group(1)) + float(m.group(2))) / 2
        else:
            nums = re.findall(r'\d+(?:\.\d+)?', s)
            if not nums: return np.nan
            v = float(nums[0])
        return v * 60 if ('hour' in s or 'hr' in s) else v

    for col in ['years_worked', 'years_at_location', 'years_lived', 'num_workers']:
        d[col + '_num'] = d[col].apply(to_years)
    d['commute_num'] = d['commute_min'].apply(to_minutes)
    agree_dummy = {'Strongly agree': 1, 'Agree': 1, 'Neutral': 0, 'Disagree': 0, 'Strongly disagree': 0}
    agree_score = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5}
    for col in ['changed_a_lot', 'good_for_longtime', 'good_for_business', 'afford_to_stay',
                'support_housing', 'connected', 'safe_at_night']:
        d[col + '_agree_dv'] = d[col].map(agree_dummy)
        d[col + '_score'] = d[col].map(agree_score)
    increased_dummy = {'Increased a lot': 1, 'Increased a little': 1, 'Stayed about the same': 0,
                       'Decreased a little': 0, 'Decreased a lot': 0}
    change_score = {'Decreased a lot': 1, 'Decreased a little': 2, 'Stayed about the same': 3,
                    'Increased a little': 4, 'Increased a lot': 5}
    for col in ['p12_customers', 'p12_sales', 'p12_space_rent', 'p12_other_costs', 'p12_own_housing', 'p12_prices']:
        d[col + '_increased_dv'] = d[col].map(increased_dummy)
        d[col + '_score'] = d[col].map(change_score)
    d['lives_here_dv'] = d['lives_here'].map({'Yes': 1, 'No': 0})
    d['rent_home_dv'] = d['home_tenure'].map({'Rent': 1, 'Own': 0})
    d['space_rented_dv'] = d['space_tenure'].map({'Rent': 1, 'Own': 0})
    d['owner_dv'] = np.where(d['role'].isna(), np.nan, (d['role'] == 'Owner').astype(float))
    d['self_admin_dv'] = d['mode'].map({'Self-administered': 1, 'Interviewer': 0})
    d['drives_alone_dv'] = np.where(d['travel_mode'].isna(), np.nan,
                                    d['travel_mode'].str.contains('Drive alone').astype(float))
    d['transit_dv'] = np.where(d['travel_mode'].isna(), np.nan,
                               d['travel_mode'].str.contains('Public transit').astype(float))
    return d

df = None
for path in ['survey_cleaned_2026.csv', '../Week13_Lab_CleaningSurveyData/survey_cleaned_2026.csv']:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded YOUR cleaned Lab 12 file: {path} ({len(df)} surveys).')
        break
if df is None:
    for path in ['CP201A_2026_survey_data.csv', '../Week13_Lab_CleaningSurveyData/CP201A_2026_survey_data.csv',
                 '../Week13_Lecture_StatisticalTests/CP201A_2026_survey_data.csv']:
        if os.path.exists(path):
            df = clean_raw_survey(pd.read_csv(path))
            print(f'No Lab 12 file found, so we cleaned the raw class file the demo way: {len(df)} surveys.')
            print("NOTE: these are the DEMO's cleaning choices, not yours. Fine for following along with the lecture.")
            print('For P/NP #11 and your final memo, finish Lab 12, save survey_cleaned_2026.csv, and rerun this cell.')
            break
if df is None:
    print('Could not find the data. Upload CP201A_2026_survey_data.csv (from bCourses) to this folder and rerun.')
else:
    # Richmond and El Cerrito combined, because El Cerrito has only a few surveys (as on November 18)
    df['city4'] = df['city'].replace({'Richmond': 'Richmond or El Cerrito', 'El Cerrito': 'Richmond or El Cerrito'})

## 2. The functions (run this cell; you don't need to change it)

- `regression_table(model)`: one row per variable with the coefficient, standard error, confidence interval, p-value, and whether it's significant at your level, plus n and R squared
- `coefficient_chart(model, labels, title)`: a dot-and-whisker chart of the coefficients, a good exhibit for a memo
- `regression_sentence(...)`: a draft sentence about one coefficient, for you to edit
- `who_is_in_the_model(df, variables)`: how many people answered every question in the model, and which variable loses the most people

**Set your confidence level once, before you look at any results.**

In [ ]:
CONFIDENCE = 0.95          # choose 0.90, 0.95, or 0.99 BEFORE looking at results
ALPHA = 1 - CONFIDENCE

def ptext(p):
    return 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'

def sig_words(p):
    return 'statistically significant' if p < ALPHA else 'not statistically significant'

def regression_table(model):
    """A tidy regression table: one row per variable, plus n and R squared."""
    level = int(round(CONFIDENCE * 100))
    ci = model.conf_int(alpha=ALPHA)
    t = pd.DataFrame({'coefficient': model.params, 'std. error': model.bse,
                      f'{level}% CI low': ci[0], f'{level}% CI high': ci[1], 'p-value': model.pvalues})
    t['significant?'] = np.where(t['p-value'] < ALPHA, f'yes, at {level}%', 'no')
    r2 = getattr(model, 'rsquared', None)
    print(f'n = {int(model.nobs)}' + (f'   R squared = {r2:.3f}' if r2 is not None else '')
          + ('   (robust standard errors)' if model.cov_type != 'nonrobust' else ''))
    return t.round(3)

def coefficient_chart(model, labels, title, xlabel='Coefficient'):
    """Dot-and-whisker chart. labels: a dictionary from the model's variable names to plain words.
    Only the variables in labels are drawn (leave out the intercept)."""
    ci = model.conf_int(alpha=ALPHA)
    names = [v for v in labels if v in model.params.index]
    est = model.params[names]
    lo, hi = ci.loc[names, 0], ci.loc[names, 1]
    fig, ax = plt.subplots(figsize=(7, 0.6 * len(names) + 1.5))
    ypos = np.arange(len(names))[::-1]
    colors = ['#E85C34' if p < ALPHA else '#5B7FA8' for p in model.pvalues[names]]
    ax.hlines(ypos, lo, hi, color=colors, linewidth=2)
    ax.scatter(est, ypos, color=colors, s=50, zorder=3)
    ax.axvline(0, color='gray', linewidth=1, linestyle='--')
    ax.set_yticks(ypos)
    ax.set_yticklabels([labels[v] for v in names])
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    print(f'For your Data Note: dots are coefficients; lines are {CONFIDENCE:.0%} confidence intervals. '
          f'Orange: significant at the {CONFIDENCE:.0%} level. n = {int(model.nobs)}.')

def regression_sentence(model, x, x_words, y_words, units, scale_words='', controls_words=None, percentage_points=False):
    """A draft sentence about one coefficient. x_words describes a one-unit change in x
    (for a dummy: 'renting one's home rather than owning it'; for a number: 'each additional year at the location')."""
    scale = 100 if percentage_points else 1
    unit = 'percentage points' if percentage_points else units
    digits = 1 if percentage_points else 2
    b = model.params[x] * scale
    lo, hi = model.conf_int(alpha=ALPHA).loc[x] * scale
    p = model.pvalues[x]
    start = f'Comparing respondents with similar {controls_words}, ' if controls_words else ''
    print(f'DRAFT SENTENCE (edit the wording): {start}{x_words} was associated with {y_words} that was '
          f'{abs(b):.{digits}f} {unit} {"higher" if b > 0 else "lower"}{scale_words} '
          f'({CONFIDENCE:.0%} CI {lo:.{digits}f} to {hi:.{digits}f}; {ptext(p)}). '
          f'This association was {sig_words(p)} at the {CONFIDENCE:.0%} level (n = {int(model.nobs)}).')

def who_is_in_the_model(data, variables):
    """A regression uses only people who answered EVERY variable in it. This shows how many that is."""
    print(f'{len(data)} surveys in the data.')
    for v in variables:
        print(f'  {v:>30}: {data[v].notna().sum()} answered')
    n = data[variables].dropna().shape[0]
    print(f'  {"answered all of them":>30}: {n}  <- the n your regression will use')

## 3. A regression line: one number X

A regression draws the straight line that comes closest to all the dots, and describes it with two numbers:

**predicted Y = intercept + slope × X**

- The **slope** (the coefficient on X) is how much predicted Y changes when X goes up by 1.
- The **intercept** is predicted Y when X is 0. Sometimes it isn't meaningful on its own (here it's the predicted score for someone who just started); it's there to place the line.

**Example:** feeling connected to the community (Y: "I feel connected to the people and community in this neighborhood," statement 2f, 1 to 5) and years worked or volunteered at this place (X: Q6). The correlation (Section 4.3 of the lecture notebook) said how *closely* they move together; the slope says *how much* Y changes per year of X.

In [ ]:
d = df[['connected_score', 'years_worked_num']].dropna()
slope, intercept = np.polyfit(d['years_worked_num'], d['connected_score'], 1)
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(d['years_worked_num'], d['connected_score'] + rng.uniform(-0.2, 0.2, len(d)), alpha=0.45, color='#1B4A82', edgecolor='white')
grid = np.linspace(0, d['years_worked_num'].max(), 50)
ax.plot(grid, intercept + slope * grid, color='#E85C34', linewidth=2)
ax.set_xlabel('Years worked or volunteered here (Q6)')
ax.set_ylabel('Feel connected (2f), 1 to 5 (jittered)')
ax.set_title(f'The line: predicted score = {intercept:.2f} + {slope:.3f} × years worked')
plt.tight_layout(); plt.show()

In [ ]:
m_simple = smf.ols('connected_score ~ years_worked_num', data=df).fit(cov_type='HC1')
regression_table(m_simple)

**Reading the table, column by column**

| Column | What it means here |
|---|---|
| `coefficient` | For `years_worked_num`: the slope. For `Intercept`: predicted Y when X is 0 (someone in their first year). |
| `std. error` | How much the coefficient would bounce around from sample to sample. Smaller is more precise. |
| `CI low`, `CI high` | The range of slopes the data are consistent with, at your confidence level. If it doesn't include 0, the coefficient is significant at that level. |
| `p-value` | The same idea as every test so far: how surprising this slope would be if the true slope were 0. |
| `n` | How many people answered both questions. |
| `R squared` | The share of the variation in Y that the line accounts for (0 to 1). With one number X, it's Pearson's r squared. |

We use **robust standard errors** (`cov_type='HC1'`). Like Welch's t-test, they don't assume that every group is equally spread out. They change the standard errors and p-values, not the coefficients.

In [ ]:
# EXERCISE #1: In a # comment, write one sentence that interprets the slope in plain words
# ("Each additional year someone has worked here goes with a connection score about ___ points higher ...").
# Then write a second sentence that says why this is an association, not a cause.

## 4. A dummy X: the difference between two groups

When X is a 0/1 dummy, "X goes up by 1" means "moving from the 0 group to the 1 group." So:

- the **intercept** is the average of Y for the 0 group, and
- the **coefficient** is the difference between the two group averages: exactly the difference a t-test compares.

**Example:** commute minutes (Y, Q9) for residents vs. people who live elsewhere (X = `lives_here_dv`, Q7).

In [ ]:
m_dummy = smf.ols('commute_num ~ lives_here_dv', data=df).fit(cov_type='HC1')
print(df.groupby('lives_here_dv')['commute_num'].agg(['count', 'mean']).round(2), '\n')
regression_table(m_dummy)

Check it: the intercept is the average for people who live elsewhere, and the intercept plus the coefficient is the average for residents. The p-value is close to the November 18 Welch t-test, because robust standard errors and Welch's test handle unequal spreads in a similar way.

## 5. A category X: dummies and the reference group

A category with several groups (like city) goes into a regression as a set of dummies, **one for each group but one**. The group left out is the **reference group**, and every coefficient compares a group with it. `C(city4)` makes the dummies for you; by default the reference group is the first one alphabetically.

You choose the reference group. Pick one that makes your comparisons easy to explain: the largest group, or the one your question is about. Here we make **Oakland** the reference with `Treatment(reference='Oakland')`.

In [ ]:
m_city = smf.ols("commute_num ~ C(city4, Treatment(reference='Oakland'))", data=df).fit(cov_type='HC1')
print(df.groupby('city4')['commute_num'].agg(['count', 'mean']).round(1), '\n')
regression_table(m_city)

Each city's coefficient is its average commute **minus** Oakland's. The names get long; `coefficient_chart` lets you give them plain labels.

In [ ]:
# EXERCISE #2: Copy the m_city line and change the reference group to 'San Francisco'.
# Which numbers change, and which stay the same? Answer in a # comment.
# (Hint: compare the intercept, the coefficients, and R squared.)

## 6. Multiple regression: "holding other things constant"

Our respondents weren't chosen at random, and groups differ in more than one way at once. Renters and owners, for example, may also differ in whether they live in the neighborhood, which city they were surveyed in, and how the survey was given. A **multiple regression** puts several X's in at once. Each coefficient then compares people who are alike on the **other** X's in the model.

**Example.** Y = agreement that long-time residents can afford to stay (`afford_to_stay_score`, 1 to 5). The X we care about is renting one's home (`rent_home_dv`). The other X's are **controls**: living in the neighborhood, years worked here, city, and survey mode (self-administered surveys gave different answers on this question on November 18).

Build it in steps and watch the renter coefficient.

In [ ]:
model_vars = ['afford_to_stay_score', 'rent_home_dv', 'lives_here_dv', 'years_worked_num', 'city4', 'self_admin_dv']
who_is_in_the_model(df, model_vars)

# Use the same people in every step, so the coefficients are comparable
d6 = df[model_vars].dropna()
step1 = smf.ols('afford_to_stay_score ~ rent_home_dv', data=d6).fit(cov_type='HC1')
step2 = smf.ols('afford_to_stay_score ~ rent_home_dv + lives_here_dv + years_worked_num', data=d6).fit(cov_type='HC1')
step3 = smf.ols("afford_to_stay_score ~ rent_home_dv + lives_here_dv + years_worked_num"
                " + C(city4, Treatment(reference='Oakland')) + self_admin_dv", data=d6).fit(cov_type='HC1')
for name, m in [('Step 1: renter only', step1), ('Step 2: + lives here, years worked', step2),
                ('Step 3: + city and survey mode', step3)]:
    print(f"{name:>38}: renter coefficient = {m.params['rent_home_dv']:+.2f}, {ptext(m.pvalues['rent_home_dv'])}, R squared = {m.rsquared:.3f}")

In [ ]:
regression_table(step3)

In [ ]:
coefficient_chart(step3,
    labels={'rent_home_dv': 'Rents home (vs. owns)',
            'lives_here_dv': 'Lives in the neighborhood',
            'years_worked_num': 'Each year worked here',
            "C(city4, Treatment(reference='Oakland'))[T.Berkeley]": 'Berkeley (vs. Oakland)',
            "C(city4, Treatment(reference='Oakland'))[T.Richmond or El Cerrito]": 'Richmond or El Cerrito (vs. Oakland)',
            "C(city4, Treatment(reference='Oakland'))[T.San Francisco]": 'San Francisco (vs. Oakland)',
            'self_admin_dv': 'Self-administered survey'},
    title='Agreement that long-time residents can afford to stay (1 to 5)',
    xlabel='Difference in average score (points on the 1 to 5 scale)')

In [ ]:
regression_sentence(step3, 'rent_home_dv', x_words='renting one\'s home rather than owning it',
                    y_words='an average agreement score', units='points', scale_words=' on the 1 to 5 scale',
                    controls_words='residence in the neighborhood, years worked, city, and survey mode')

**How to read the steps.** If the renter coefficient stays about the same from Step 1 to Step 3, the renter-owner difference isn't explained by the controls. If it shrinks toward 0, part of the difference was really about those other things. If it grows, the controls were hiding some of it. Any of these is a finding worth a sentence in your memo.

**What "holding constant" can't do.** It only accounts for the variables you put in. Anything left out (income, age, how long someone has lived in the Bay Area) can still be behind a coefficient. So a coefficient from survey data is an **association, after accounting for the variables in the model**, never proof of a cause.

In [ ]:
# EXERCISE #3: Add one more control to step3 that you think might matter (for example, owner_dv,
# space_rented_dv, or drives_alone_dv). Use d6 = df[...].dropna() again with the new variable in the list.
# Does the renter coefficient change? Answer in a # comment.

## 7. A 0/1 outcome: coefficients in percentage points

If Y is a dummy, a linear regression still works. It's called a **linear probability model**: each coefficient, times 100, is a difference in **percentage points**. That makes it easy to explain, and it lines up with the Z-test.

**Example.** Y = own rent or mortgage went up in the past 12 months (`p12_own_housing_increased_dv`). X = rents one's home. This is the P/NP #11 example (renters vs. owners). Here we check whether the gap holds when we account for city, living in the neighborhood, and survey mode.

In [ ]:
vars7 = ['p12_own_housing_increased_dv', 'rent_home_dv', 'lives_here_dv', 'city4', 'self_admin_dv']
who_is_in_the_model(df, vars7)
d7 = df[vars7].dropna()
lpm1 = smf.ols('p12_own_housing_increased_dv ~ rent_home_dv', data=d7).fit(cov_type='HC1')
lpm2 = smf.ols("p12_own_housing_increased_dv ~ rent_home_dv + lives_here_dv"
               " + C(city4, Treatment(reference='Oakland')) + self_admin_dv", data=d7).fit(cov_type='HC1')
print(f"Renter gap, no controls:   {lpm1.params['rent_home_dv'] * 100:+.1f} percentage points")
print(f"Renter gap, with controls: {lpm2.params['rent_home_dv'] * 100:+.1f} percentage points\n")
regression_table(lpm2)

In [ ]:
regression_sentence(lpm2, 'rent_home_dv', x_words='renting one\'s home rather than owning it',
                    y_words='a likelihood of saying their own rent or mortgage went up',
                    units='', controls_words='residence in the neighborhood, city, and survey mode',
                    percentage_points=True)

### 7.1 Optional: logistic regression

Published research usually uses **logistic regression** (logit) for a 0/1 outcome, because a straight line can predict shares below 0 or above 100 percent. Logit coefficients are in "log odds," which are hard to explain, so researchers often report **average marginal effects**: the average change in the chance of Y = 1, which reads like the linear model's percentage points. When most predicted shares are between about 20 and 80 percent, the two usually tell the same story. Run this to compare:

In [ ]:
logit = smf.logit("p12_own_housing_increased_dv ~ rent_home_dv + lives_here_dv"
                  " + C(city4, Treatment(reference='Oakland')) + self_admin_dv", data=d7).fit(disp=0)
ame = logit.get_margeff(at='overall').summary_frame()
print(f"Renter gap, linear probability model:  {lpm2.params['rent_home_dv'] * 100:+.1f} percentage points")
print(f"Renter gap, logit (average marginal effect): {ame.loc['rent_home_dv', 'dy/dx'] * 100:+.1f} percentage points")

## 8. Checks before you trust a regression

**1. Check who's in the model.** A regression drops anyone who skipped *any* variable in it. Run `who_is_in_the_model` and report the n. If one variable loses a lot of people (like `years_lived_num`, which only residents answered), think about leaving it out.

**2. Enough people for the number of X's.** A common rule of thumb is at least 10 to 20 people for every X (each city dummy counts as one). With about 200 people, five to eight X's is plenty. More X's than that makes every coefficient noisier.

**3. Small groups.** A category with only a few people (El Cerrito alone) gives a coefficient you can't trust. Combine groups, as we did with `city4`.

**4. Two X's that measure the same thing.** If two X's are very closely related (for example, renting one's home and owning one's business), the model can't tell them apart, and both coefficients get noisy. Check the correlation between your X's.

**5. Look at the leftovers.** The **residuals** are each person's actual Y minus the predicted Y. Plot them against the predictions: a shapeless cloud is good; a strong pattern or a few huge residuals is worth a look.

In [ ]:
# Check 4: how closely related are the X's? (Values above about 0.7 are worth a second look.)
print(d6[['rent_home_dv', 'lives_here_dv', 'years_worked_num', 'self_admin_dv']].corr().round(2), '\n')

# Check 5: residuals vs. predicted values for step3
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(step3.fittedvalues, step3.resid, alpha=0.4, color='#1B4A82')
ax.axhline(0, color='gray', linestyle='--')
ax.set_xlabel('Predicted score'); ax.set_ylabel('Residual (actual minus predicted)')
ax.set_title('Residuals for the Section 6 model')
plt.tight_layout(); plt.show()

With a 1 to 5 score (or a 0/1 outcome), the residuals fall in diagonal stripes, because Y can only take a few values. That's expected. What you're looking for is a few people far from everyone else, or a curve.

## 9. Your turn: your own regression

**Step 1. Start from a hypothesis** you've already tested (P/NP #11): one Y and one main X.

**Step 2. Choose two to five controls:** things that might be related to both your X and your Y. City (`C(city4)`) and survey mode (`self_admin_dv`) are good defaults for our survey. Write one line on why each is there.

**Step 3. Check who's in the model** with `who_is_in_the_model`.

**Step 4. Run it in steps:** your main X alone, then with the controls, on the same people. Watch the main coefficient.

**Step 5. Make your exhibit:** a table (save it as a CSV and format it in Word or Excel) or a coefficient chart. Give every variable a plain-language label.

**Step 6. Write it up** (Section 10).

**How to fill in the cell:** put numbers, scores, and 0/1 dummies in the `CONTROLS` list, and categories (like `city4`) in `CATEGORY_CONTROLS` with the reference group you want. The cell builds the regression for you and prints it, so you can check it.

**If you get an error:** a `KeyError` or `PatsyError` usually means a variable name is misspelled (copy it from the Step 2 list in the lecture notebook); a `NameError` means Sections 1 and 2 need to be run again; a reference group that isn't spelled exactly as in the data (for example `'oakland'`) gives an error that mentions `Treatment`. Still stuck? Email the teaching team with the error and the cell, or bring it to office hours.

In [ ]:
# EXERCISE #4: your regression
# Step 1, my hypothesis (Y, main X, and the direction I expect):
# Step 2, my controls and why:

Y = 'afford_to_stay_score'                 # your outcome: a number, a 1 to 5 score, or a 0/1 dummy
MAIN_X = 'rent_home_dv'                    # your main explanatory variable
CONTROLS = ['self_admin_dv', 'lives_here_dv']        # controls that are numbers, scores, or 0/1 dummies
CATEGORY_CONTROLS = {'city4': 'Oakland'}             # controls that are categories, with the reference group you choose
                                                     # (use {} if you have none)

# You don't need to change anything below this line.
terms = [MAIN_X] + CONTROLS + [f"C({c}, Treatment(reference='{ref}'))" for c, ref in CATEGORY_CONTROLS.items()]
print('Your model:', Y, '~', ' + '.join(terms), '\n')

# Step 3: who's in the model
plain = [Y, MAIN_X] + CONTROLS + list(CATEGORY_CONTROLS)
who_is_in_the_model(df, plain)
mine_d = df[plain].dropna()

# Step 4: main X alone, then with controls, on the same people
mine1 = smf.ols(f'{Y} ~ {MAIN_X}', data=mine_d).fit(cov_type='HC1')
mine2 = smf.ols(f'{Y} ~ ' + ' + '.join(terms), data=mine_d).fit(cov_type='HC1')
print(f'\nMain X alone:      {mine1.params[MAIN_X]:+.3f} ({ptext(mine1.pvalues[MAIN_X])})')
print(f'With your controls: {mine2.params[MAIN_X]:+.3f} ({ptext(mine2.pvalues[MAIN_X])})\n')
regression_table(mine2)

In [ ]:
# Step 5: save the table for your memo, and/or make a chart. Change the file name and labels.
regression_table(mine2).to_csv('my_regression_table.csv')
print('Saved my_regression_table.csv. Download it, then format it in Word or Excel with plain labels.')

## 10. Writing it up

A regression finding in your memo needs the same pieces as any tested finding, plus a few more.

**In the text:** the hypothesis; the main coefficient in plain words with its units (points on the 1 to 5 scale, minutes, or percentage points); the confidence interval or p-value at your chosen level; what you held constant; whether the coefficient changed when you added the controls; and that it's an association, not a cause.

**In the exhibit:** plain-language labels (not `rent_home_dv`), the reference group for every category, the n, and R squared.

**In the Data Note:** "Source: CP201A class survey, 2026. Linear regression (ordinary least squares) with robust standard errors; n = [n] respondents who answered every question in the model. Y is [how coded]; [X] is [how coded]; reference groups: [list]. Respondents were not a random sample, so results describe the people we surveyed."

**A model paragraph** (from Section 6; check the numbers against your own output):

> Renters were [more / less] likely than homeowners to agree that long-time residents can afford to stay. Comparing respondents with similar residence in the neighborhood, years worked, city, and survey mode, renting one's home was associated with an average agreement score [___] points [higher / lower] on the 1 to 5 scale (95% CI [___] to [___]; p = [___]; n = [___]; Table X). The gap [changed / barely changed] when we added these controls, which suggests [___]. Because the survey was not a random sample and the model can account only for the variables we measured, this is an association, not evidence that renting causes the difference.

In [ ]:
# EXERCISE #5: Write two or three sentences about YOUR regression, as # comments, following the model paragraph.
# Use regression_sentence(...) to get started, then edit it into your own words.